## Hybrid Retriever- Combining Dense And Sparse Retriever

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from langchain.chat_models.base import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


/home/aniruddha/Projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

dense_retriever = dense_vectorstore.as_retriever()

In [3]:
### Sparse Retriever(BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3 ##top- k documents to retriever

# Step 4: combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weight=[0.7, 0.3]
)



In [4]:
# Step 5: Query and get results
query = "How can I build application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [5]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template(
    """
    Answer the question based on the context below.

    Context:
    {context}

    Question: {input}
    """
)

# Step 6: LLM
llm = init_chat_model("openai:gpt-4.1")

In [6]:
# Step 7: Create stuff Docuemnt Chain
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [7]:
# Step 8: create full RAG chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=document_chain
)

In [10]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 To build an app using LLMs (Large Language Models), you can follow these steps:

1. **Choose a Framework:**  
   Use a framework like LangChain, which is designed to help build LLM applications.

2. **Select Your LLM:**  
   Pick a large language model (such as OpenAI GPT-4, Anthropic Claude, etc.) accessible via API.

3. **Integrate LangChain:**  
   Use LangChain to connect your LLM to your application's backend. LangChain provides tools to:
   - Manage prompts
   - Chain multiple LLM calls together
   - Develop agentic applications (apps that can make decisions or take actions based on user input)
   - Interface with tools like retrievers and databases

4. **Add Retrieval Functionality:**  
   If your app needs to search external documents or data:
   - Use a retriever in LangChain (several options available)
   - Store and search your data using a vector database like Pinecone for semantic search

5. **Build the App Logic:**  
   - Define how the app interacts with users